# Notebook 1: Setup Environment - Spark + Nessie

**Objectif**: Initialiser l'environnement de travail et se connecter au catalogue de données Nessie.

## Concepts clés

- **Apache Spark**: Moteur de traitement distribué pour les données volumineuses
- **Project Nessie**: Catalogue de données avec contrôle de version (comme Git pour les données)
- **Apache Iceberg**: Format de table ACID avec time travel

## Architecture

```
Spark Application
        │
        ├── spark_catalog (catalogue par défaut)
        │
        └── nessie (notre catalogue versionné)
                │
                ├── bronze (données brutes)
                ├── silver (données nettoyées)
                └── gold (agrégats métier)
```

**Prérequis**: Nessie doit être démarré avec `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`

---
## 1. Import et configuration

In [ ]:
import sys
import os

# Ajouter le répertoire src au chemin Python
project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Project root: {project_root}")

## 2. Création de la session Spark

La session Spark est configurée pour:
- Se connecter à Nessie (http://localhost:19120)
- Utiliser Iceberg comme format de table
- Se connecter à S3 pour le stockage des données

In [ ]:
from lakehouse.spark_session import get_spark

# Créer la session Spark
spark = get_spark("demo-lakehouse")

print("✅ Session Spark créée avec succès!")
print(f"   Version Spark: {spark.version}")

## 3. Vérifier la connexion à Nessie

In [ ]:
# Lister tous les catalogues disponibles
print("=== Catalogues disponibles ===")
spark.sql("SHOW CATALOGS").show(truncate=False)

## 4. Créer les namespaces (bronze, silver, gold)

Dans un lakehouse, on organise les tables par **niveau de qualité** - c'est l'architecture **Medallion**:

| Namespace | Usage |
|-----------|------|
| **bronze** | Données brutes, telles que reçues de la source |
| **silver** | Données nettoyées, standardisées, dédupliquées |
| **gold** | Données agrégées, prêtes pour la consommation |

In [ ]:
# Créer les namespaces s'ils n'existent pas
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("✅ Namespaces créés dans Nessie")

In [ ]:
# Vérifier les namespaces créés
print("=== Namespaces Nessie ===")
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

## 5. Vérifier les tables existantes

Au début, les namespaces sont vides. Après avoir exécuté les notebooks suivants, des tables apparaîtront ici.

In [ ]:
print("=== Tables dans Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("\n=== Tables dans Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("\n=== Tables dans Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

## 6. Test de connexion S3

Les données sont stockées dans S3. Vérifions que nous pouvons y accéder.

In [ ]:
from lakehouse.settings import AWS_BUCKET

print(f"Bucket S3 configuré: {AWS_BUCKET}")

# Essayer de lire le fichier source
try:
    test_df = spark.read.format("csv").option("header", "true").load(f"s3a://{AWS_BUCKET}/raw/sales/superstore_sales.csv")
    print(f"\n✅ Fichier source accessible: {test_df.count():,} enregistrements")
except Exception as e:
    print(f"\n⚠️ Impossible d'accéder au fichier source: {e}")
    print("   Utilisez 'python scripts/upload_to_s3.py' pour uploader les données sur S3.")

## 7. Résumé

Notre environnement est maintenant prêt! Voici ce que nous avons:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         ENVIRONNEMENT LAKEHOUSE PRÊT                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  Spark Session    ✅ Configuré                               ║
║  Nessie Catalog  ✅ Connecté                                ║
║  Iceberg Format  ✅ Activé                                  ║
║  S3 Storage      ✅ Configuré                               ║
║                                                            ║
║  Namespaces:                                               ║
║    • nessie.bronze  (données brutes)                        ║
║    • nessie.silver  (données nettoyées)                     ║
║    • nessie.gold    (agrégats métier)                       ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

print("\n➡️ Prochaine étape: Exécuter '02_ingestion_bronze.ipynb'")